In [1]:
import os
import glob
import numpy as np
import pandas as pd
from PIL import Image

import torch
from torchvision import transforms, models

torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

main_model_preprocess = transforms.Compose(
    [
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def find_test_images_dir():
    candidates = [
        "/kaggle/input/cassava-leaf-disease-classification/test_images",
        "/kaggle/data/cassava-leaf-disease-classification/test_images",
        "/kaggle/input/cassava-leaf-disease-classification/cassava-leaf-disease-classification/test_images",
        "/kaggle/working/cassava-leaf-disease-classification/test_images",
    ]
    for p in candidates:
        if os.path.isdir(p):
            return p
    raise FileNotFoundError(
        "Could not find test_images directory in expected Kaggle paths."
    )


def find_train_images_dir():
    candidates = [
        "/kaggle/input/cassava-leaf-disease-classification/train_images",
        "/kaggle/data/cassava-leaf-disease-classification/train_images",
        "/kaggle/input/cassava-leaf-disease-classification/cassava-leaf-disease-classification/train_images",
        "/kaggle/working/cassava-leaf-disease-classification/train_images",
    ]
    for p in candidates:
        if os.path.isdir(p):
            return p
    return None


def find_train_csv_path():
    candidates = [
        "/kaggle/input/cassava-leaf-disease-classification/train.csv",
        "/kaggle/data/cassava-leaf-disease-classification/train.csv",
        "/kaggle/input/cassava-leaf-disease-classification/cassava-leaf-disease-classification/train.csv",
        "/kaggle/working/cassava-leaf-disease-classification/train.csv",
        "/kaggle/input/train.csv",
        "/kaggle/data/train.csv",
    ]
    for p in candidates:
        if os.path.isfile(p):
            return p
    return None


def find_sample_submission_path():
    candidates = [
        "/kaggle/input/cassava-leaf-disease-classification/sample_submission.csv",
        "/kaggle/data/cassava-leaf-disease-classification/sample_submission.csv",
        "/kaggle/input/cassava-leaf-disease-classification/cassava-leaf-disease-classification/sample_submission.csv",
        "/kaggle/working/cassava-leaf-disease-classification/sample_submission.csv",
        "/kaggle/input/sample_submission.csv",
        "/kaggle/data/sample_submission.csv",
    ]
    for p in candidates:
        if os.path.isfile(p):
            return p
    return None


def find_checkpoint_path():
    """
    Keep existing robust checkpoint search logic.
    """
    original = (
        "/kaggle/input/resnet50_70_512x512/pytorch/default/1/Resnet50_70_512x512.pth"
    )
    if os.path.isfile(original):
        return original

    patterns = [
        "/kaggle/input/**/Resnet50_70_512x512.pth",
        "/kaggle/input/**/resnet50*.pth",
        "/kaggle/input/**/*.pth",
        "/kaggle/input/**/*.pt",
    ]
    for pat in patterns:
        matches = glob.glob(pat, recursive=True)
        if matches:
            matches = sorted(set(matches))
            return matches[0]
    return None


def _ten_crop_224_from_256(img_pil: Image.Image):
    """
    Deterministic multi-view crops (used for prototypes and test-time).
    """
    img = img_pil.resize((256, 256), resample=Image.BILINEAR)
    crops = []
    coords = [
        (0, 0),
        (256 - 224, 0),
        (0, 256 - 224),
        (256 - 224, 256 - 224),
        ((256 - 224) // 2, (256 - 224) // 2),
    ]
    for x0, y0 in coords:
        crops.append(img.crop((x0, y0, x0 + 224, y0 + 224)))
    flips = [c.transpose(Image.FLIP_LEFT_RIGHT) for c in crops]
    return crops + flips  # 10 crops


class _ProtoDataset(torch.utils.data.Dataset):
    def __init__(self, rows, images_dir):
        self.rows = rows
        self.images_dir = images_dir
        self.normalize = transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        image_id, label = self.rows[idx]
        img_path = os.path.join(self.images_dir, image_id)
        img = Image.open(img_path).convert("RGB")

        crops = _ten_crop_224_from_256(img)
        xs = []
        for c in crops:
            t = transforms.ToTensor()(c)
            t = self.normalize(t)
            xs.append(t)
        x = torch.stack(xs, dim=0)  # [V=10, 3, 224, 224]
        y = int(label)
        return x, y


class CosineLinear(torch.nn.Module):
    def __init__(self, in_features: int, out_features: int, scale: float = 22.0):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.weight = torch.nn.Parameter(torch.empty(out_features, in_features))
        torch.nn.init.normal_(self.weight, mean=0.0, std=0.01)
        self.scale = float(scale)

    def forward(self, x):
        x = torch.nn.functional.normalize(x, dim=1)
        w = torch.nn.functional.normalize(self.weight, dim=1)
        return self.scale * (x @ w.t())


def init_fc_from_train_prototypes(
    model_5cls: torch.nn.Module,
    preprocess,
    device,
    max_images_per_class: int = 256,
    batch_size: int = 32,
    scale: float = 22.0,
    cov_eps: float = 1e-3,
    proto_shrink: float = 0.05,
):
    """
    Inference-only prototype head initialization (core logic preserved).

    Bugfix (runtime): ensure fc parameters are moved to `device` before using them to
    compute logits / normalized weights; otherwise cosine head can stay on CPU while
    extracted features are on GPU, causing a matmul device mismatch.
    """
    train_csv_path = find_train_csv_path()
    train_images_dir = find_train_images_dir()
    if train_csv_path is None or train_images_dir is None:
        return {"used": False, "reason": "train.csv or train_images dir not found"}

    df = pd.read_csv(train_csv_path)
    if "image_id" not in df.columns or "label" not in df.columns:
        return {"used": False, "reason": "train.csv missing required columns"}

    df["image_id"] = df["image_id"].astype(str)
    df["label"] = df["label"].astype(int)
    df = df.sort_values(["label", "image_id"]).reset_index(drop=True)

    rows = []
    counts = []
    for lbl in range(5):
        sub = df[df["label"] == lbl].head(max_images_per_class)
        counts.append(int(len(sub)))
        for r in sub.itertuples(index=False):
            img_path = os.path.join(train_images_dir, r.image_id)
            if os.path.isfile(img_path):
                rows.append((r.image_id, int(r.label)))

    if len(rows) == 0 or min(counts) == 0:
        return {
            "used": False,
            "reason": f"insufficient images per class (counts={counts})",
        }

    # Ensure model and head are on the same device used for feature extraction.
    model_5cls.to(device)
    model_5cls.eval()

    feat_extractor = torch.nn.Sequential(*(list(model_5cls.children())[:-1])).to(device)
    feat_extractor.eval()

    ds = _ProtoDataset(rows, train_images_dir)

    try:
        cpu_cnt = os.cpu_count() or 2
    except Exception:
        cpu_cnt = 2
    num_workers = min(4, max(0, cpu_cnt - 1))

    dl = torch.utils.data.DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
    )

    in_features = (
        model_5cls.fc.in_features if hasattr(model_5cls.fc, "in_features") else 2048
    )

    feats_sum = torch.zeros((5, in_features), dtype=torch.float32, device=device)
    feats_cnt = torch.zeros((5,), dtype=torch.long, device=device)

    g_sum = torch.zeros((in_features,), dtype=torch.float32, device=device)
    g_sq_sum = torch.zeros((in_features,), dtype=torch.float32, device=device)
    g_cnt = torch.zeros((), dtype=torch.long, device=device)

    with torch.no_grad():
        for xb, yb in dl:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            B, V = xb.shape[0], xb.shape[1]
            xb2 = xb.view(B * V, *xb.shape[2:])  # [B*V, 3, 224, 224]
            f = feat_extractor(xb2).flatten(1)  # [B*V, 2048]
            f = torch.nn.functional.normalize(f, dim=1)
            f = f.view(B, V, -1)

            # Per-view logits, then average logits across views (same as test-time).
            lv = model_5cls.fc(f.reshape(B * V, -1)).view(B, V, 5)  # [B,V,5]
            l = lv.mean(dim=1)  # [B,5]

            p = torch.softmax(l, dim=1)  # [B,5]

            W = torch.nn.functional.normalize(model_5cls.fc.weight, dim=1)  # [5,2048]
            f_logit = p @ W  # [B,2048]
            f_logit = torch.nn.functional.normalize(f_logit, dim=1)

            g_sum += f_logit.sum(dim=0)
            g_sq_sum += (f_logit * f_logit).sum(dim=0)
            g_cnt += B

            for c in range(5):
                m = yb == c
                if m.any():
                    feats_sum[c] += f_logit[m].sum(dim=0)
                    feats_cnt[c] += int(m.sum().item())

    cnt_cpu = feats_cnt.detach().cpu().numpy().tolist()
    if min(cnt_cpu) == 0:
        return {
            "used": False,
            "reason": f"insufficient usable images per class (counts={cnt_cpu})",
        }

    protos = feats_sum / feats_cnt.unsqueeze(1).float()

    g_mean = g_sum / g_cnt.clamp(min=1).float()
    g_var = (g_sq_sum / g_cnt.clamp(min=1).float()) - (g_mean * g_mean)
    g_var = torch.clamp(g_var, min=cov_eps)
    inv_std = torch.rsqrt(g_var)

    protos = (protos - g_mean.unsqueeze(0)) * inv_std.unsqueeze(0)

    if proto_shrink is not None and float(proto_shrink) > 0.0:
        global_proto = protos.mean(dim=0, keepdim=True)
        protos = (1.0 - float(proto_shrink)) * protos + float(
            proto_shrink
        ) * global_proto

    protos = torch.nn.functional.normalize(protos, dim=1)

    with torch.no_grad():
        if hasattr(model_5cls.fc, "weight"):
            model_5cls.fc.weight.copy_(protos)
        if hasattr(model_5cls.fc, "bias") and model_5cls.fc.bias is not None:
            model_5cls.fc.bias.zero_()
        if hasattr(model_5cls.fc, "scale"):
            model_5cls.fc.scale = float(scale)

    return {
        "used": True,
        "counts": cnt_cpu,
        "max_images_per_class": max_images_per_class,
        "batch_size": batch_size,
        "scale": scale,
        "views_per_image": 10,
        "proto_aug": "Deterministic 10-crop: Resize(256)->(5-crop + hflip) averaged",
        "head": "CosineLinear",
        "num_workers": num_workers,
        "whitening": f"global diag whitening (eps={cov_eps})",
        "proto_space": "logit-aligned (softmax(logits) @ normalized_fc_weight)",
        "proto_shrink": float(proto_shrink),
    }


ckpt_path = find_checkpoint_path()

model = None
if ckpt_path is not None:
    obj = torch.load(ckpt_path, map_location=device)
    if isinstance(obj, torch.nn.Module):
        model = obj
    elif isinstance(obj, dict):
        state_dict = None
        if "state_dict" in obj and isinstance(obj["state_dict"], dict):
            state_dict = obj["state_dict"]
        elif "model_state_dict" in obj and isinstance(obj["model_state_dict"], dict):
            state_dict = obj["model_state_dict"]
        elif all(isinstance(k, str) and torch.is_tensor(v) for k, v in obj.items()):
            state_dict = obj

        backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        backbone.fc = torch.nn.Linear(backbone.fc.in_features, 5)

        if state_dict is not None:
            cleaned = {}
            for k, v in state_dict.items():
                nk = k.replace("module.", "") if k.startswith("module.") else k
                cleaned[nk] = v
            backbone.load_state_dict(cleaned, strict=False)
        model = backbone
    else:
        model = None

prototype_init_info = None
if model is None:
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    model.fc = CosineLinear(model.fc.in_features, 5, scale=22.0)

    prototype_init_info = init_fc_from_train_prototypes(
        model,
        main_model_preprocess,
        device,
        max_images_per_class=4096,
        batch_size=64 if torch.cuda.is_available() else 32,
        scale=22.0,
        cov_eps=1e-3,
        proto_shrink=0.05,
    )

model.to(device)
model.eval()

test_images_dir = find_test_images_dir()
sample_sub_path = find_sample_submission_path()

print("Device:", device)
print("Test images dir:", test_images_dir)
print("Sample submission path:", sample_sub_path)
print("Checkpoint used:", ckpt_path)
print("Prototype init info (None if ckpt used):", prototype_init_info)



Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 107MB/s]


Device: cuda
Test images dir: /kaggle/input/cassava-leaf-disease-classification/test_images
Sample submission path: /kaggle/input/cassava-leaf-disease-classification/sample_submission.csv
Checkpoint used: None
Prototype init info (None if ckpt used): {'used': True, 'counts': [939, 1901, 2091, 4096, 2267], 'max_images_per_class': 4096, 'batch_size': 64, 'scale': 22.0, 'views_per_image': 10, 'proto_aug': 'Deterministic 10-crop: Resize(256)->(5-crop + hflip) averaged', 'head': 'CosineLinear', 'num_workers': 4, 'whitening': 'global diag whitening (eps=0.001)', 'proto_space': 'logit-aligned (softmax(logits) @ normalized_fc_weight)', 'proto_shrink': 0.05}


In [2]:
if sample_sub_path is not None:
    sample_df = pd.read_csv(sample_sub_path)
    image_ids = sample_df["image_id"].astype(str).tolist()
else:
    image_ids = sorted(
        [f for f in os.listdir(test_images_dir) if f.lower().endswith(".jpg")]
    )


class _TestDatasetTTA(torch.utils.data.Dataset):
    """
    Deterministic 10-crop test-time augmentation and logits averaging.
    """

    def __init__(self, image_ids, images_dir):
        self.image_ids = image_ids
        self.images_dir = images_dir
        self.normalize = transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_name = self.image_ids[idx]
        img_path = os.path.join(self.images_dir, image_name)
        img = Image.open(img_path).convert("RGB")
        crops = _ten_crop_224_from_256(img)
        xs = []
        for c in crops:
            t = transforms.ToTensor()(c)
            t = self.normalize(t)
            xs.append(t)
        x = torch.stack(xs, dim=0)  # [V=10, 3, 224, 224]
        return x


test_ds = _TestDatasetTTA(image_ids, test_images_dir)

try:
    cpu_cnt = os.cpu_count() or 2
except Exception:
    cpu_cnt = 2
num_workers = min(4, max(0, cpu_cnt - 1))

test_dl = torch.utils.data.DataLoader(
    test_ds,
    batch_size=16,  # smaller because each sample carries 10 views
    shuffle=False,
    num_workers=num_workers,
    pin_memory=torch.cuda.is_available(),
    drop_last=False,
)

prediction = []
with torch.no_grad():
    for xb in test_dl:
        xb = xb.to(device, non_blocking=True)  # [B, V, 3, 224, 224]
        B, V = xb.shape[0], xb.shape[1]
        xb2 = xb.view(B * V, *xb.shape[2:])  # [B*V, 3, 224, 224]
        out = model(xb2)  # [B*V, 5]
        out = out.view(B, V, -1).mean(dim=1)  # average logits across views -> [B, 5]
        pred = torch.argmax(out, dim=1).detach().cpu().numpy().astype(np.int64).tolist()
        prediction.extend(pred)

print("Predictions computed:", len(prediction))



Predictions computed: 2676


In [3]:
submission = pd.DataFrame({"image_id": image_ids, "label": prediction})
submission["label"] = submission["label"].astype(int)

if submission.isna().any().any():
    raise ValueError("Submission contains NaNs.")
if submission.shape[0] == 0:
    raise ValueError("Empty submission; no test images found or inference failed.")

submission.to_csv("submission.csv", index=False)

print(submission.head())
print(f"Wrote submission.csv with {len(submission)} rows.")

         image_id  label
0  1234294272.jpg      2
1  1234332763.jpg      4
2  1234375577.jpg      4
3  1234555380.jpg      1
4  1234571117.jpg      3
Wrote submission.csv with 2676 rows.
